In [12]:
import time
import os
import pandas as pd
import ScraperFC as sfc

In [25]:
output_dir = "../../data/raw/v2"

In [2]:
ss=sfc.Sofascore()

In [32]:
Seasons=["25/26","24/25","23/24"]
Leagues = ["England Premier League","Spain La Liga","Germany Bundesliga","Italy Serie A","France Ligue 1"]
Positions = ["Goalkeepers", "Defenders", "Midfielders", "Forwards"]

In [34]:
for season in Seasons:
    master_df=[]
    for league in Leagues:
        for pos in Positions:
            
            print(f"Scraping {league} {season}")
            
            try:
                df = ss.scrape_player_league_stats(
                    year=season,
                    league=league,
                    accumulation="total",
                    selected_positions=[pos],
                )
                df["league"] = league
                df["season"] = season
                df["position"] = pos
                master_df.append(df)
                print(f"{len(df)} players, {len(df.columns)} columns")
                
            except Exception as e:
                print(f"fail: {e}")
            time.sleep(5)
            
    if master_df:
        raw_season_df = pd.concat(master_df, ignore_index=True)
        position_combine = (
        raw_season_df.groupby(['player', 'team'])['position']
        .apply(lambda x: ", ".join(sorted(set(x))))
        .reset_index()
    )
        numeric_cols = raw_season_df.select_dtypes(include=['number']).columns.tolist()
        if 'player id' in numeric_cols: numeric_cols.remove('player id')
        if 'team id' in numeric_cols: numeric_cols.remove('team id')
        
        aggregated_stats = raw_season_df.groupby(['player', 'team'])[numeric_cols].first().reset_index()
        final_season_df = pd.merge(aggregated_stats, position_combine, on=['player', 'team'], how='inner')
        metadata_df = raw_season_df.drop_duplicates(subset=['player', 'team'])
        metadata_cols = ['player', 'team', 'player id', 'team id', 'league', 'season']
        final_season_df = pd.merge(final_season_df, metadata_df[metadata_cols], on=['player', 'team'], how='inner')
        
        file_path = f"{output_dir}/sofascore_top5_{season[0:2]+season[3:]}season.csv"            
        final_season_df.to_csv(file_path, index=False)
        print(f"Saved: {len(final_season_df)} total rows for {season} season")
        
        
    else:
        
        
        print(f"No data was scraped for the {season} season.")
        

Scraping England Premier League 25/26
40 players, 117 columns
Scraping England Premier League 25/26
178 players, 117 columns
Scraping England Premier League 25/26
215 players, 117 columns
Scraping England Premier League 25/26
104 players, 117 columns
Scraping Spain La Liga 25/26
39 players, 117 columns
Scraping Spain La Liga 25/26
203 players, 117 columns
Scraping Spain La Liga 25/26
240 players, 117 columns
Scraping Spain La Liga 25/26
118 players, 117 columns
Scraping Germany Bundesliga 25/26
30 players, 117 columns
Scraping Germany Bundesliga 25/26
159 players, 117 columns
Scraping Germany Bundesliga 25/26
184 players, 117 columns
Scraping Germany Bundesliga 25/26
126 players, 117 columns
Scraping Italy Serie A 25/26
46 players, 117 columns
Scraping Italy Serie A 25/26
169 players, 117 columns
Scraping Italy Serie A 25/26
217 players, 117 columns
Scraping Italy Serie A 25/26
154 players, 117 columns
Scraping France Ligue 1 25/26
39 players, 117 columns
Scraping France Ligue 1 25/26


In [26]:
print(sorted(final_season_df.columns.tolist()))


['accurateChippedPasses', 'accurateCrosses', 'accurateCrossesPercentage', 'accurateFinalThirdPasses', 'accurateLongBalls', 'accurateLongBallsPercentage', 'accurateOppositionHalfPasses', 'accurateOwnHalfPasses', 'accuratePasses', 'accuratePassesPercentage', 'aerialDuelsWon', 'aerialDuelsWonPercentage', 'aerialLost', 'appearances', 'assists', 'attemptPenaltyMiss', 'attemptPenaltyPost', 'attemptPenaltyTarget', 'ballRecovery', 'bigChancesCreated', 'bigChancesMissed', 'blockedShots', 'cleanSheet', 'clearances', 'countRating', 'crossesNotClaimed', 'directRedCards', 'dispossessed', 'dribbledPast', 'duelLost', 'errorLeadToGoal', 'errorLeadToShot', 'expectedAssists', 'expectedGoals', 'fouls', 'freeKickGoal', 'goalConversionPercentage', 'goalKicks', 'goals', 'goalsAssistsSum', 'goalsConceded', 'goalsConcededInsideTheBox', 'goalsConcededOutsideTheBox', 'goalsFromInsideTheBox', 'goalsFromOutsideTheBox', 'goalsPrevented', 'groundDuelsWon', 'groundDuelsWonPercentage', 'headedGoals', 'highClaims', 'h